# 개별종목 조합C — LogisticRegression

`기본모델/01.LogisticRegression.ipynb`과 같은 `models.logistic.build_logistic_baseline`을 가져오고
조합C 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.logistic import build_logistic_baseline  # noqa: E402

MODEL_NAME = 'LogisticRegression'
MODEL_BUILDER = build_logistic_baseline


In [2]:
# 2. 조합C의 피처 값만 지정합니다.
import json

COMBINATION = 'C'
FEATURE_COLUMNS = (
    'ret_5',
    'overnight_gap',
    'intraday_return',
    'close_location',
    'bb_position',
    'rsi_14',
    'volume_z_20',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)
combination_report = report["combinations"].get(COMBINATION)
if combination_report is None:
    print("아직 실측 결과가 없습니다. 아래 공통 실행 명령으로 조합을 평가하세요.")
else:
    panel = combination_report["panel"]
    print("학습 기간:", panel["first_date"], "~", panel["last_date"])
    print("학습 행·종목:", panel["model_rows"], panel["stocks"])
    folds = pd.DataFrame(combination_report["outer_fold_results"])
    model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
    fold_columns = [
        "fold", "selected_class_weight", "train_dates", "valid_start", "valid_end",
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, fold_columns].round(4))
    metric_columns = [
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


조합C 피처: ('ret_5', 'overnight_gap', 'intraday_return', 'close_location', 'bb_position', 'rsi_14', 'volume_z_20')
학습 기간: 20110127 ~ 20240822
학습 행·종목: 159900 157


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,balanced_accuracy,mcc,pr_auc_macro_ovr,down_recall,core_harmonic_mean
0,1,balanced,750,20140217,20140514,0.3130,0.5012,-0.1882,0.3128,0.3528,0.0233,0.3609,0.3968,0.3366
1,2,balanced,980,20150123,20150421,0.3235,0.3978,-0.0743,0.3163,0.3423,0.0090,0.3425,0.4470,0.3534
2,3,balanced,1210,20151228,20160328,0.3189,0.3762,-0.0572,0.2984,0.3342,0.0008,0.3381,0.4481,0.3441
3,4,balanced,1439,20161202,20170228,0.2981,0.4617,-0.1636,0.2870,0.3500,0.0169,0.3535,0.5254,0.3432
4,5,balanced,1669,20171113,20180207,0.3331,0.3901,-0.0570,0.3168,0.3531,0.0301,0.3582,0.4654,0.3611
5,6,balanced,1899,20181024,20190118,0.3404,0.3725,-0.0321,0.3216,0.3546,0.0312,0.3806,0.4778,0.3685
6,7,balanced,2129,20190930,20191224,0.3139,0.4781,-0.1643,0.3075,0.3700,0.0497,0.3611,0.5318,0.3606
7,8,balanced,2359,20200902,20201130,0.3667,0.3476,0.0190,0.3593,0.3730,0.0574,0.3773,0.4790,0.3948
8,9,balanced,2589,20210806,20211105,0.3334,0.3914,-0.0580,0.3330,0.3591,0.0382,0.3705,0.2480,0.2990
9,10,balanced,2818,20220714,20221012,0.3433,0.3454,-0.0021,0.3434,0.3506,0.0266,0.3449,0.2878,0.3226


,OOS 폴드 평균
accuracy,0.3326
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,-0.0643
macro_f1,0.3250
balanced_accuracy,0.3541
mcc,0.0292
pr_auc_macro_ovr,0.3586
down_recall,0.4195
core_harmonic_mean,0.3497


재실행 명령: python scripts/run_stock_model_experiment.py
